# Gemma 4 E2B Architecture Enumeration

Phase 1 Task B: Enumerate Gemma 4 E2B module structure to fill `notes/architecture.md`.

**Run on Kaggle** (T4 GPU, 16GB). E2B loads in ~10GB in bfloat16. No HF token required — model is mounted locally.

**Kaggle model**: https://www.kaggle.com/models/google/gemma-4/transformers/gemma-4-e2b
Add via "Add Input" in the Kaggle notebook editor.

**Outputs needed:**
- Full module tree (`named_modules`)
- Config dump (attention pattern, num_kv_shared_layers, PLE config)
- HF class names and exact attribute names
- PLE parameter shapes from state dict
- RoPE theta values (local + global)
- Forward pass trace to confirm PLE data flow

In [ ]:
import os

# Confirm Kaggle mount path — adjust if listdir shows a different subpath
kaggle_base = '/kaggle/input/gemma-4/transformers/gemma-4-e2b/1'
print("Kaggle input dirs:", os.listdir('/kaggle/input/'))
if os.path.exists(kaggle_base):
    print(f"Model path confirmed: {kaggle_base}")
    print("Contents:", os.listdir(kaggle_base))
else:
    # Fallback: find the actual path
    for root, dirs, files in os.walk('/kaggle/input/'):
        if any(f.endswith('.safetensors') or f == 'config.json' for f in files):
            print(f"Found model files at: {root}")
            break

!pip install -q transformers accelerate

In [ ]:
# Kaggle local path — no HF token needed
MODEL_ID = '/kaggle/input/gemma-4/transformers/gemma-4-e2b/1'

# If the path check above found a different path, update this manually
print(f"Using model path: {MODEL_ID}")

## 1. Config inspection (no weights loaded)

In [ ]:
from transformers import AutoConfig

MODEL_ID = "google/gemma-4-E2B-it"
cfg = AutoConfig.from_pretrained(MODEL_ID)

print("=== Config type ===")
print(type(cfg))
print()
print("=== Full config ===")
print(cfg)

In [ ]:
# Extract key fields — adjust attribute names if they differ
interesting = [
    'num_hidden_layers',
    'hidden_size',
    'num_attention_heads',
    'num_key_value_heads',
    'intermediate_size',
    'sliding_window',
    'attention_bias',
    'rope_theta',
    # PLE / shared KV — names TBD
    'num_kv_shared_layers',
    'use_per_layer_embeddings',
    'per_layer_embedding_size',
    # Attention type pattern
    'attention_types',
    'layer_types',
    'sliding_window_pattern',
    'attn_logit_softcapping',
    'query_pre_attn_scalar',
]

print("=== Key config fields ===")
for field in interesting:
    val = getattr(cfg, field, '(not found)')
    print(f"  {field}: {val}")

print()
print("=== All config keys ===")
print(sorted(cfg.to_dict().keys()))

## 2. Load model and enumerate modules

In [ ]:
import torch
from transformers import AutoModelForCausalLM

# Load in bfloat16 on T4 — uses ~10GB
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
print(f"Loaded: {type(model).__name__}")
print(f"Config class: {type(model.config).__name__}")

In [ ]:
# Full module tree — paste output into notes/architecture.md
print("=== named_modules() ===")
for name, mod in model.named_modules():
    print(f"{name}: {type(mod).__name__}")

In [ ]:
# Per-layer breakdown: attention type and PLE presence
print("=== Per-layer module types ===")
# Access the decoder layers — adjust path if needed
layers = model.model.layers
print(f"Num layers: {len(layers)}")
print()

for i, layer in enumerate(layers):
    attn_type = type(layer.self_attn).__name__
    children = [n for n, _ in layer.named_children()]
    print(f"Layer {i:2d}: attn={attn_type}, children={children}")

In [ ]:
# PLE parameters in state dict
print("=== PLE-related parameters ===")
sd = model.state_dict()
ple_keys = [k for k in sd.keys() if 'per_layer' in k.lower() or 'ple' in k.lower()]
if ple_keys:
    for k in ple_keys:
        print(f"  {k}: {sd[k].shape}")
else:
    print("  (no keys matching 'per_layer' or 'ple' — check module tree for correct name)")

print()
print("=== Shared KV parameters ===")
kv_keys = [k for k in sd.keys() if 'shared' in k.lower() or 'kv_cache' in k.lower()]
if kv_keys:
    for k in kv_keys:
        print(f"  {k}: {sd[k].shape}")
else:
    print("  (no keys matching 'shared'/'kv_cache' — shared KV may be activation-based)")

In [ ]:
# Inspect layer 0 in detail
print("=== Layer 0 full detail ===")
print(layers[0])
print()
print("=== Layer 0 attention ===")
print(layers[0].self_attn)
print()
# Check if there's a sliding window attribute
attn = layers[0].self_attn
print("Attention attributes:", [a for a in dir(attn) if not a.startswith('_')])

In [ ]:
# Identify local vs global layers by inspecting sliding_window or is_sliding attributes
print("=== Per-layer attention type ===")
for i, layer in enumerate(layers):
    attn = layer.self_attn
    is_local = getattr(attn, 'is_sliding', None)
    sliding_window = getattr(attn, 'sliding_window', None)
    layer_type = getattr(attn, 'layer_type', None)
    print(f"Layer {i:2d}: is_sliding={is_local}, sliding_window={sliding_window}, layer_type={layer_type}")

In [ ]:
# RoPE theta values — local attention layers may use 10000, global may use 1000000
# Check both config-level and per-layer overrides
print("=== RoPE theta ===")
print(f"  config.rope_theta: {getattr(model.config, 'rope_theta', '(not found)')}")
print(f"  config.rope_local_base_freq: {getattr(model.config, 'rope_local_base_freq', '(not found)')}")
print(f"  config.rope_global_base_freq: {getattr(model.config, 'rope_global_base_freq', '(not found)')}")

# Check per-layer rotary embedding if distinct
for i, layer in enumerate(layers[:2]):
    attn = layer.self_attn
    rotary = getattr(attn, 'rotary_emb', None)
    if rotary is not None:
        print(f"  Layer {i} rotary_emb type: {type(rotary).__name__}")
        print(f"    base: {getattr(rotary, 'base', '(not found)')}")
        print(f"    theta: {getattr(rotary, 'theta', '(not found)')}")

# Also check the config dict directly
cfg_dict = model.config.to_dict()
rope_keys = [k for k in cfg_dict if 'rope' in k.lower() or 'rotary' in k.lower()]
print(f"\n  All RoPE config keys: {rope_keys}")
for k in rope_keys:
    print(f"    {k}: {cfg_dict[k]}")

## 3. Forward pass trace — confirm PLE data flow

In [ ]:
# Final: print summary for copy-paste into architecture.md
print("=== SUMMARY FOR architecture.md ===")
print(f"Model class: {type(model).__name__}")
print(f"Config class: {type(model.config).__name__}")
print(f"Num layers: {len(layers)}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num heads: {model.config.num_attention_heads}")
print(f"Num KV heads: {model.config.num_key_value_heads}")
print(f"Intermediate size (d_mlp): {getattr(model.config, 'intermediate_size', '(check named_modules)')}")
print(f"Vocab size: {model.config.vocab_size}")
print()
print("=== PLE state dict keys ===")
sd = model.state_dict()
ple_keys = [k for k in sd.keys() if any(x in k.lower() for x in ['per_layer', 'layer_scalar'])]
for k in ple_keys[:30]:  # cap at 30 to avoid flooding output
    print(f"  {k}: {sd[k].shape}")
print()
print("=== Attention type per layer ===")
for i, layer in enumerate(layers):
    attn = layer.self_attn
    is_local = getattr(attn, 'is_sliding', getattr(attn, 'is_local', None))
    layer_type = getattr(attn, 'layer_type', None)
    print(f"  Layer {i:2d}: is_sliding={is_local}, layer_type={layer_type}")

In [ ]:
# Final: print summary for copy-paste into architecture.md
print("=== SUMMARY FOR architecture.md ===")
print(f"Model class: {type(model).__name__}")
print(f"Config class: {type(model.config).__name__}")
print(f"Num layers: {len(layers)}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num heads: {model.config.num_attention_heads}")
print(f"Num KV heads: {model.config.num_key_value_heads}")
print()
print("PLE state dict keys (copy to architecture.md):")
for k in [k for k in sd.keys() if any(x in k.lower() for x in ['per_layer', 'ple', 'altup', 'lauren'])]:
    print(f"  {k}: {sd[k].shape}")